# Phase 6E: ConvNeXt + PhoBERT + Gated Cross-Modal + Auto-Weight
**Phase 6 | Promising Combinations**
This notebook tests Alternative Candidate 4 from Phase 6.
- Image: ConvNeXt | Text: PhoBERT | Fusion: Gated Cross-Modal | Loss: Auto-Weight | Seed: 42


### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13471, done.
remote: Counting objects: 100% (342/342), done.
remote: Compressing objects: 100% (209/209), done.
remote: Total 13471 (delta 250), reused 222 (delta 133), pack-reused 13129 (from 1)
Receiving objects: 100% (13471/13471), 873.24 MiB | 20.08 MiB/s, done.
Resolving deltas: 100% (488/488), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

total 1420
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 24 14:46 ..
drwxr-xr-x  2 root root 1437696 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_060E_convnext_phobert_gatedcrossmodal_autoweight'

BEST_IMAGE_EXP_ID = 'EXP_011_image_only_convnext_meanpool_mse'
BEST_TEXT_EXP_ID  = 'EXP_030B_bestimage_phobert_concat_mse'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts: {DRIVE_EXP_PATH}')


Artifacts: /content/drive/MyDrive/SE365/experiments/EXP_060E_convnext_phobert_gatedcrossmodal_autoweight


### STEP 5: Load pretrained weights

In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_TEXT_EXP_ID}/best_model_train_text.pth', './checkpoints/best_model_train_text.pth')
print(f'Loaded text from {BEST_TEXT_EXP_ID}')
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image from {BEST_IMAGE_EXP_ID}')


Loaded text from EXP_030B_bestimage_phobert_concat_mse
Loaded image from EXP_011_image_only_convnext_meanpool_mse


### STEP 6: Train

In [ ]:
!python main.py \
  --mode train_fusion \
  --fusion_type gated_cross \
  --text_model_name vinai/phobert-base-v2 \
  --image_model_name convnext_base_in22k \
  --loss_fn auto_weight \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_060E_convnext_phobert_gatedcrossmodal_autoweight \
  --exp_dir ./experiments


====== MODE: TRAIN_FUSION ======
Using device: cuda
Seed: 42 | Experiment: EXP_060E_convnext_phobert_gatedcrossmodal_autoweight
config.json: 100% 678/678 [00:00<00:00, 3.18MB/s]
vocab.txt: 100% 895k/895k [00:00<00:00, 98.6MB/s]
bpe.codes: 100% 1.14M/1.14M [00:00<00:00, 116MB/s]
tokenizer.json: 100% 3.13M/3.13M [00:00<00:00, 148MB/s]
Loaded timm processor for convnext_base_in22k
pytorch_model.bin: 100% 540M/540M [00:05<00:00, 95.9MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 18499.03it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from di

### STEP 7: Evaluate on Test Set
Evaluate the best model on the unseen test set to report final metrics and generate plots.

In [ ]:
!python test.py \
  --mode train_fusion \
  --fusion_type gated_cross \
  --text_model_name vinai/phobert-base-v2 \
  --image_model_name convnext_base_in22k \
  --loss_fn auto_weight \
  --exp_id EXP_060E_convnext_phobert_gatedcrossmodal_autoweight \
  --exp_dir ./experiments


====== TESTING: TRAIN_FUSION ======
Device: cuda
Test samples: 600
Loading weights: 100% 197/197 [00:00<00:00, 18045.34it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name convnext_base_in22k to current convnext_base.fb_in22k.
  model = crea

### STEP 8: Save to Drive + print metrics


In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

# --- VALIDATION METRICS ---
with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results (Validation) ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")

# --- TEST METRICS ---
with open(f'./experiments/{EXP_ID}/test_metrics.json') as f:
    t = json.load(f)

print(f'\n=== {EXP_ID} Results (Test) ===')
print()
print("             MAE      RMSE      R2")
print(f"  food     : {t['mae_food']:.4f}   {t['rmse_food']:.4f}   {t['r2_food']:.4f}")
print(f"  price    : {t['mae_price']:.4f}   {t['rmse_price']:.4f}   {t['r2_price']:.4f}")
print(f"  atmos    : {t['mae_atmos']:.4f}   {t['rmse_atmos']:.4f}   {t['r2_atmos']:.4f}")
print(f"  service  : {t['mae_service']:.4f}   {t['rmse_service']:.4f}   {t['r2_service']:.4f}")
print(f"  overall  : {t['mae_overall']:.4f}   {t['rmse_overall']:.4f}   {t['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {t['mean_mae']:.4f}")
print(f"  aspect_mae : {t['aspect_mae']:.4f}")
print(f"  overall_mae: {t['overall_mae']:.4f}")



=== EXP_060E_convnext_phobert_gatedcrossmodal_autoweight Results (Validation) ===
Loss (val)   : 2.2396

             MAE      RMSE      R2
  food     : 1.1205   1.5029   0.5708
  price    : 1.1838   1.5899   0.4341
  atmos    : 1.1908   1.5375   0.3909
  service  : 1.1858   1.5803   0.5129
  overall  : 0.9435   1.2561   0.6125

  mean_mae   : 1.1248
  aspect_mae : 1.1702
  overall_mae: 0.9435

=== EXP_060E_convnext_phobert_gatedcrossmodal_autoweight Results (Test) ===

             MAE      RMSE      R2
  food     : 1.0525   1.4617   0.6133
  price    : 1.1212   1.4863   0.4736
  atmos    : 1.1935   1.5665   0.3516
  service  : 1.0998   1.4976   0.5271
  overall  : 0.9098   1.2019   0.6330

  mean_mae   : 1.0754
  aspect_mae : 1.1168
  overall_mae: 0.9098
